# M044_2024_12_04_09_30

Session: M044_2024_12_04_09_30

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../..")

import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
import seaborn as sns
import pyaldata as pyal

from tools.dsp.preprocessing import preprocess
import tools.viz.rasters as rt
from tools.params import Params
from tools.viz.dimensionality import plot_VAF
from tools.dataTools import get_data_array
from tools.viz.rasters import plot_heatmap_raster

from tools.dimensionality.participation import participation_ratio, pca_pr
from tools.params import colors
from tools.dimensionality.cca import canoncorr



In [ ]:
# TODO: add example data to the repo and run on that
data_dir = "/data/bnd-data/raw/M044/M044_2024_12_04_09_30"
fname = os.path.join(data_dir, "M044_2024_12_04_09_30_pyaldata.mat")

In [ ]:
# load TrialData .mat file into a DataFrame
df = pyal.mat2dataframe(fname, shift_idx_fields=True)

## Preprocessing from utils

In [ ]:
df_ = preprocess(df, only_trials=False)
areas = ["M1", "Dls"]
df_["M1_rates"] = [df_["all_rates"][i][:,300:] for i in range(len(df_))]
df_["Dls_rates"] = [df_["all_rates"][i][:,0:300] for i in range(len(df_))]

## Plotting

In [ ]:
df_ = pyal.select_trials(df_, "idx_trial_end > 30365")  # Remove first 5 minutes because the switch was off

In [ ]:
df_.head()

In [ ]:
df_.values_Sol_direction == 1

- Trial 90 has some nice spikes

In [ ]:
areas = ["M1", "Dls"]
df_["M1_rates"] = [df_["all_rates"][i][:,300:] for i in range(len(df_))]
df_["Dls_rates"] = [df_["all_rates"][i][:,0:300] for i in range(len(df_))]

In [ ]:
fig, axes = plt.subplots(1, figsize=(15, 5), sharey=True)

# axes.imshow(rates.T, aspect="auto")
area="Dls"
plot_heatmap_raster(pyal.select_trials(df_, df_.trial_name == 'trial')[50:75], area=area,ax=axes, show=False, add_sol_onset=True)
plt.show()

# rates.shape

In [ ]:
df__ = pyal.select_trials(df_, df.values_Sol_duration == 150)
for trial in range(8):
    fig, axes = plt.subplots(1, len(Params.sol_dir_to_level.keys()), figsize=(15, 5))
    axes = rt.plot_fr_raster(df__, axes, Params.sol_dir_to_level.keys(), trial=trial, area='Dls')
plt.show()
# 


In [ ]:
df__ = pyal.select_trials(df_, df.values_Sol_duration == 150)
for trial in range(8):
    fig, axes = plt.subplots(1, len(Params.sol_dir_to_level.keys()), figsize=(15, 3))
    axes = rt.plot_fr_raster(df__, axes, Params.sol_dir_to_level.keys(), trial=trial, area='M1')
plt.show()
# 


## VAF

In [ ]:
data_list = [pyal.select_trials(df_, df_.trial_name == 'trial')]
areas = ["M1", "Dls"]
n_components = None
epoch = None
model = "pca"

In [ ]:
# VAF for each area in areas list, averaged across sessions in data_list, with shaded errorbars.
fig, ax = plt.subplots()
ax = plot_VAF(ax = ax, data_list = data_list, areas = areas, n_components = n_components, epoch = epoch, model = model, show=False)

In [ ]:
df_intertrial = preprocess(df, only_trials=False)
df_intertrial = pyal.select_trials(df_intertrial, df_intertrial.trial_name == "intertrial")


In [ ]:
df_trials = pyal.select_trials(df_, df_.trial_name == 'trial')
trial_data = pyal.concat_trials(df_trials, f"{area}_rates")
print(trial_data.shape)

In [ ]:
# TODO: Refactor into function
from sklearn.decomposition import PCA

def plot_participation_ratio_per_session(df, areas, epoch = None, trial_query = None, intertrial_query = None, title=None):
    results = {area: {'free': [], 'inter': [], 'trial': []} for area in areas}

    df_trials = pyal.select_trials(df, df.trial_name == 'trial')
    df_trials_motion = df_trials[df_trials['idx_motion'].apply(lambda x: np.any(x < df_trials.idx_sol_on[0]))]


    df_intertrials = pyal.select_trials(df, df.trial_name == 'intertrial')
    df_free = pyal.select_trials(df, df.trial_name == 'free')


    if epoch is not None:
        df_trials = pyal.restrict_to_interval(df_trials, epoch_fun=epoch)

    if trial_query is not None:
        print("Applying query")
        print(len(df_trials))
        df_trials = pyal.select_trials(df_trials, trial_query)
        print(len(df_trials))


    for area in areas:
        free_data = pyal.concat_trials(df_free, f"{area}_rates")
        results[area]['free'].append(pca_pr(free_data))

        trial_data = pyal.concat_trials(df_trials, f"{area}_rates")
        results[area]['trial'].append(pca_pr(trial_data))

        intertrial_data = pyal.concat_trials(df_intertrials, f"{area}_rates")
        results[area]['inter'].append(pca_pr(intertrial_data))

    

    fig, axes = plt.subplots(1, len(areas), sharey=True)

    for i, area in enumerate(areas):
        data = pd.DataFrame({
            'PR': results[area]['free'] + results[area]['inter'] + results[area]['trial'],
            'Condition': ['free'] * len(results[area]['free']) + ['inter'] * len(results[area]['inter']) + ['trial'] * len(results[area]['trial'])
        })
        sns.stripplot(data=data, x='Condition', y='PR', s=8, color='blue', ax=axes[i])
        axes[i].set_title(area)

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()
    print(results)


areas = ["M1", "Dls"]
# plot_participation_ratio_per_session(df_, areas, trial_query="idx_trial_end > 30000")


In [ ]:
from tools.dimensionality.participation import plot_participation_ratio_per_session

areas = ["M1", "Dls"]
fig, axes = plt.subplots(1, len(areas), sharey=True)
_ = plot_participation_ratio_per_session(df_, areas, axes, epoch=None)

In [ ]:
n_components = None
epoch=None
model = "pca"

df_trials = pyal.select_trials(df_, df_.trial_name == 'trial')
df_trials = pyal.select_trials(df_trials, "idx_trial_end > 30000")
df_trials_motion = df_trials[df_trials['idx_motion'].apply(lambda x: np.any(x < df_trials.idx_sol_on[0]))]

df_intertrials = pyal.select_trials(df_, df_.trial_name == 'intertrial')
df_free = pyal.select_trials(df_, df_.trial_name == 'free')

fig, axes = plt.subplots(1, len(areas), figsize=(10, 5), sharey='all', sharex='all')
for area, ax in zip(areas, axes):
    ax1 = plot_VAF(ax = ax, data_list = [df_trials], areas = area, n_components = n_components, epoch = epoch, model = model, show=False)
    ax1 = plot_VAF(ax = ax, data_list = [df_free], areas = area, n_components = n_components, epoch = epoch, model = model, show=False, linestyle='--')
    ax1 = plot_VAF(ax = ax, data_list = [df_intertrials], areas = area, n_components = n_components, epoch = epoch, model = model, show=False, linestyle=':')


    ax.set_title(f"{area}")



plt.show()

In [ ]:
results = {area: {'0': [], '1': []} for area in areas}
sol_levels = [0, 1]

window_times = np.arange(start=0.1, stop=1.9, step=0.1)

for area in areas:
    for sol_level in sol_levels:
        for window_time in window_times:
            perturb_epoch = pyal.generate_epoch_fun(
                start_point_name="idx_sol_on",
                rel_start=int(-0.5 / Params.BIN_SIZE),
                rel_end=int(window_time / Params.BIN_SIZE),
            )

            df__ = pyal.restrict_to_interval(df_trials_motion, epoch_fun=perturb_epoch)
            df_specfic_sol_opening = pyal.select_trials(df__, df__.sol_level_id == sol_level)
            rates = pyal.concat_trials(df_specfic_sol_opening, f"{area}_rates")
            results[area][str(sol_level)].append(pca_pr(rates))


fig, axes = plt.subplots(1, len(areas), figsize=(10, 5), sharey='all')
for area, ax in zip(areas, axes):

    ax.plot(window_times, results[area]['0'], color=getattr(colors, area), label=f"{area}_upper", marker='o', linestyle='-')
    ax.plot(window_times, results[area]['1'], color=getattr(colors, area), label=f"{area}_lower", marker='P', linestyle='--')
    ax.legend()
    # ax.set_xticklabels(['Upper', 'Lower'])
    ax.set_xlabel('Time window (s)')
    ax.set_ylabel('PR')


In [ ]:
results = {area: [] for area in areas}
sol_levels = [0, 1]

df_trials_motion = df_trials[df_trials['idx_motion'].apply(lambda x: np.any(x < df_trials.idx_sol_on[0]))]

for area in areas:
    for sol_level in sol_levels:
        df_specfic_sol_opening = pyal.select_trials(df_trials_motion, df_trials_motion.sol_level_id == sol_level)
        rates = pyal.concat_trials(df_specfic_sol_opening, f"{area}_rates")
        results[area].append(pca_pr(rates))

fig, ax = plt.subplots(1, figsize=(3, 7))
for area in areas:
    ax.plot(sol_levels, results[area], color=getattr(colors, area), label=area, marker='o', linestyle='-')
    ax.legend()
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Upper', 'Lower'])
    ax.set_xlabel('Solenoid Level')
    ax.set_ylabel('PR')

In [ ]:
results = {area: [] for area in areas}
sol_openings = np.unique(df_trials.values_Sol_duration)


for area in areas:
    for sol_opening in sol_openings:
        df_specfic_sol_opening = pyal.select_trials(df_trials_motion, df_trials_motion.values_Sol_duration == sol_opening)
        rates = pyal.concat_trials(df_specfic_sol_opening, f"{area}_rates")
        results[area].append(pca_pr(rates))

fig, ax = plt.subplots(1)
for area in areas:
    ax.plot(sol_openings, results[area], color=getattr(colors, area), label=area, marker='o', linestyle='-')
    ax.legend()
    ax.set_xlabel('Solenoid opening time')
    ax.set_ylabel('PR')


In [ ]:
results = {area: [] for area in areas}
sols_ipsi_contra = [0, 1]

df_trials_motion = df_trials[df_trials['idx_motion'].apply(lambda x: np.any(x < df_trials.idx_sol_on[0]))]

for area in areas:
    for sol_ipsi_contra in sols_ipsi_contra:
        df_specfic_sol_opening = pyal.select_trials(df_trials_motion, df_trials_motion.sol_contra_ipsi == sol_ipsi_contra)
        rates = pyal.concat_trials(df_specfic_sol_opening, f"{area}_rates")
        results[area].append(pca_pr(rates))

fig, ax = plt.subplots(1, figsize=(3, 7))
for area in areas:
    ax.plot(sol_levels, results[area], color=getattr(colors, area), label=area, marker='o', linestyle='-')
    ax.legend()
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Ipsi', 'Contra'])
    ax.set_xlabel('Solenoid Direction')
    ax.set_ylabel('PR')

### Trials

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA

areas = ["M1", "Dls"]
dfs = []
axes = []
category = "values_Sol_direction"
trial_types = ['trial']
epoch = None
n_components = 10

df_trials = pyal.select_trials(df_, df_.trial_name == 'trial')
df_trials_motion = df_trials[df_trials['idx_motion'].apply(lambda x: np.any(x < df_trials.idx_sol_on[0]))]

df_int = df_trials_motion[:-1]

timepoint = df_int['idx_sol_on'][0]
# Define subplot grid dimensions
n_rows = n_components  # Rows: trial types
n_cols = len(areas)        # Columns: areas
targets = np.unique(df_trials_motion[category])


fig, axes = plt.subplots(n_components, n_cols, figsize=(7, 7), sharex='all')
df__ = pyal.select_trials(df_int, f"trial_name == 'trial'")
# Ensure axes is a 2D array for easy indexing
axes = np.array(axes).reshape(n_rows, n_cols)

# Loop through areas (columns) and trial types (rows)
for col, area in enumerate(areas):
    for row in range(n_components):
        rates = np.concatenate(df__[area+'_rates'].values, axis=0)  # Shape: (239 trials, 15 timepoints, 87 units)

        # Fit PCA model
        rates_model = PCA(n_components=n_components, svd_solver='full').fit(rates)
        
        # Apply PCA to the dataframe
        df___ = pyal.apply_dim_reduce_model(df__, rates_model, area+'_rates', '_pca')

        # Select the correct subplot
        ax = axes[row, col]

        # Loop through targets and plot averaged trials
        for tar in targets:
            df____ = pyal.select_trials(df___, df___[category] == tar)
            ex = pyal.get_sig_by_trial(df____, '_pca')
            ex = np.mean(ex, axis=2)[:, :n_components]  # Reduce to first 3 PCA components
            ax.plot(ex[:, row])
            

        # Titles and labels
        ax.axvline(x = timepoint,color = 'r',linestyle="--" )
        if row==0:
            ax.set_title(f"{area}")
        if col==0:
            ax.set_ylabel(f"PC {row}")
    ax.set_xlabel("Timebins (30ms)")
        # ax.set_zlabel("PC3")

# ex = pyal.get_sig_by_trial(df_trials_motion, '_pca')
# ex = np.mean(ex, axis=2)[:, :n_components]  # Reduce to first 3 PCA components
# axes[-1, 0].plot()

# plt.tight_layout()
plt.show()

### Intertrials

In [ ]:
pyal.select_trials(df_, "trial_name == 'intertrial'").head()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA

dfs = []
axes = []
category = "trial_length"
trial_types = ['trial']
epoch = None
n_components = 10

df_intertrials = pyal.select_trials(df_, "trial_name == 'intertrial'")
trial_lengths = np.unique(df_intertrials.trial_length.values)




n_rows = n_components  # Rows: trial types
n_cols = len(areas)        # Columns: areas


fig, axes = plt.subplots(n_components, n_cols, figsize=(7, 7), sharex='all')
# # df__ = pyal.select_trials(df_int, f"trial_name == 'trial'")

df___ = df_intertrials
# # Ensure axes is a 2D array for easy indexing
axes = np.array(axes).reshape(n_rows, n_cols)


# # Loop through areas (columns) and trial types (rows)
for col, area in enumerate(areas):
    for row in range(n_components):
        rates = np.concatenate(df___[area+'_rates'].values, axis=0)  # Shape: (239 trials, 15 timepoints, 87 units)

        # Fit PCA model
        rates_model = PCA(n_components=n_components, svd_solver='full').fit(rates)
        
        # Apply PCA to the dataframe
        df___ = pyal.apply_dim_reduce_model(df___, rates_model, area+'_rates', '_pca')

        # Select the correct subplot
        ax = axes[row, col]

        # Loop through targets and plot averaged trials
        for trial_length in trial_lengths:
            df_tmp = pyal.select_trials(df___, df___[category] == trial_length)
            ex = pyal.get_sig_by_trial(df_tmp, '_pca')
            ex = np.mean(ex, axis=2)[:, :n_components]  # Reduce to first 3 PCA components
            ax.plot(ex[:, row])            


        if row==0:
            ax.set_title(f"{area}")
        if col==0:
            ax.set_ylabel(f"PC {row}")
    ax.set_xlabel("Timebins (30ms)")


# plt.tight_layout()
plt.show()

In [ ]:
df_trials_motion.columns

### Pooling upper and lower level. Upper -> Orange

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA

dfs = []
axes = []
category = "values_Sol_direction"
trial_types = ['trial']
epoch = None
n_components = 10

df_trials = pyal.select_trials(df_, df_.trial_name == 'trial')
df_trials_motion = df_trials[df_trials['idx_motion'].apply(lambda x: np.any(x < df_trials.idx_sol_on[0]))]

df_int = df_trials_motion[:-1]

timepoint = df_int['idx_sol_on'][0]
# Define subplot grid dimensions
n_rows = n_components  # Rows: trial types
n_cols = len(areas)        # Columns: areas
targets = np.unique(df_trials_motion[category])


fig, axes = plt.subplots(n_components, n_cols, figsize=(7, 7), sharex='all')
df__ = pyal.select_trials(df_int, f"trial_name == 'trial'")
# Ensure axes is a 2D array for easy indexing
axes = np.array(axes).reshape(n_rows, n_cols)

# Loop through areas (columns) and trial types (rows)
for col, area in enumerate(areas):
    for row in range(n_components):
        rates = np.concatenate(df__[area+'_rates'].values, axis=0)  # Shape: (239 trials, 15 timepoints, 87 units)

        # Fit PCA model
        rates_model = PCA(n_components=n_components, svd_solver='full').fit(rates)
        
        # Apply PCA to the dataframe
        df___ = pyal.apply_dim_reduce_model(df__, rates_model, area+'_rates', '_pca')

        # Select the correct subplot
        ax = axes[row, col]

        # Loop through targets and plot averaged trials
        for tar in targets:
            df____ = pyal.select_trials(df___, df___[category] == tar)
            level = 'upper' if Params.sol_dir_to_level[tar] == 0 else 'lower'
            ex = pyal.get_sig_by_trial(df____, '_pca')
            ex = np.mean(ex, axis=2)[:, :n_components]  # Reduce to first 3 PCA components
            ax.plot(ex[:, row], color=getattr(colors, level))
            

        # Titles and labels
        ax.axvline(x = timepoint,color = 'r',linestyle="--" )
        if row==0:
            ax.set_title(f"{area}")
        if col==0:
            ax.set_ylabel(f"PC {row}")
    ax.set_xlabel("Timebins (30ms)")
        # ax.set_zlabel("PC3")

# plt.tight_layout()
plt.show()

### Pooling contra and ipsi. Contra --> RED

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA

dfs = []
axes = []
category = "values_Sol_direction"
trial_types = ['trial']
epoch = None
n_components = 10

df_trials = pyal.select_trials(df_, df_.trial_name == 'trial')
df_trials_motion = df_trials[df_trials['idx_motion'].apply(lambda x: np.any(x < df_trials.idx_sol_on[0]))]

df_int = df_trials_motion[:-1]

timepoint = df_int['idx_sol_on'][0]
# Define subplot grid dimensions
n_rows = n_components  # Rows: trial types
n_cols = len(areas)        # Columns: areas
targets = np.unique(df_trials_motion[category])


fig, axes = plt.subplots(n_components, n_cols, figsize=(7, 7), sharex='all')
df__ = pyal.select_trials(df_int, f"trial_name == 'trial'")
# Ensure axes is a 2D array for easy indexing
axes = np.array(axes).reshape(n_rows, n_cols)

# Loop through areas (columns) and trial types (rows)
for col, area in enumerate(areas):
    for row in range(n_components):
        rates = np.concatenate(df__[area+'_rates'].values, axis=0)  # Shape: (239 trials, 15 timepoints, 87 units)

        # Fit PCA model
        rates_model = PCA(n_components=n_components, svd_solver='full').fit(rates)
        
        # Apply PCA to the dataframe
        df___ = pyal.apply_dim_reduce_model(df__, rates_model, area+'_rates', '_pca')

        # Select the correct subplot
        ax = axes[row, col]

        # Loop through targets and plot averaged trials
        for tar in targets:
            df____ = pyal.select_trials(df___, df___[category] == tar)
            side = 'ipsi' if Params.sol_dir_to_contra_ipse[tar] == 0 else 'contra'
            ex = pyal.get_sig_by_trial(df____, '_pca')
            ex = np.mean(ex, axis=2)[:, :n_components]  # Reduce to first 3 PCA components
            ax.plot(ex[:, row], color=getattr(colors, side))
            

        # Titles and labels
        ax.axvline(x = timepoint,color = 'r',linestyle="--" )
        if row==0:
            ax.set_title(f"{area}")
        if col==0:
            ax.set_ylabel(f"PC {row}")
    ax.set_xlabel("Timebins (30ms)")
        # ax.set_zlabel("PC3")

# plt.tight_layout()
plt.show()

In [ ]:
def histogram_deviation(data, num_bins=10):
    """
    Computes the deviation of a histogram from being perfectly homogeneous.
    
    Parameters:
        data (list or np.array): The data points to be histogrammed.
        num_bins (int): Number of bins in the histogram.
    
    Returns:
        dict: Contains Chi-Square Statistic and Mean Absolute Deviation (MAD).
    """
    # Compute histogram
    hist, bin_edges = np.histogram(data, bins=num_bins)
    
    # Expected frequency for uniform distribution
    expected_freq = len(data) / num_bins
    
    # Compute Chi-Square Statistic
    chi_square_stat = np.sum(((hist - expected_freq) ** 2) / expected_freq)
    return chi_square_stat


df_trials = pyal.select_trials(df_, df.trial_name == "trial")


plt.figure(figsize=(8, 5))
plt.hist(df_trials['values_Sol_direction'].values, edgecolor='black', alpha=0.75)

# Labels and title
plt.xlabel('Solenoid activated')
plt.ylabel('Count')
chi_square_stat = histogram_deviation(df_trials['values_Sol_direction'].values)
plt.title(f'Distribution of Solenoid activations. Chi-square: {chi_square_stat: .2f}')

chi_over_time = []
window_size = 100  # 5 minutes, 75 4s trials
window_starts = np.arange(0, len(df_trials), step=window_size)
for window_start in window_starts:
    df_tmp = df_trials[window_start: window_start + window_size]
    data = df_tmp['values_Sol_direction'].values
    chi_over_time.append(histogram_deviation(data))

    # plt.figure(figsize=(8, 5))
    # plt.hist(data, edgecolor='black', alpha=0.75)

    # # Labels and title
    # plt.xlabel('Solenoid activated')
    # plt.ylabel('Count')
    # chi_square_stat = histogram_deviation(data)
    # plt.title(f'Distribution of Solenoid activations. Chi-square: {chi_square_stat: .2f}')

plt.figure(figsize=(8, 5))
plt.plot(window_starts, chi_over_time)
# Labels and title
plt.xlabel('Trial')
plt.ylabel('Chi-Squared')
plt.title(f'Chi-squared over time. Window size: {window_size}')


In [ ]:
np.unique(df_trials.values_Sol_direction)

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable


df_trials = pyal.select_trials(df_, df.trial_name == "trial")


counts_mat = []
window_size = 225  # 5 minutes, 75 4s trials
window_starts = np.arange(0, len(df_trials), step=window_size)
for window_start in window_starts:
    df_tmp = df_trials[window_start: window_start + window_size]
    data = df_tmp['values_Sol_direction'].values
    counts, edges = np.histogram(data, bins=11)
    counts_mat.append(counts / np.sum(counts))


counts_mat = np.vstack(counts_mat)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(counts_mat.T)
ax.set_ylabel("Solenoid")
ax.set_xlabel("15 min Time bins")

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im, cax=cax)



### CCAs

In [ ]:
from tools.dimensionality.cca import get_ccs_between_two_areas
areas = ["M1", "Dls"]
n_components = 10

perturb_epoch = pyal.generate_epoch_fun(
        start_point_name="idx_sol_on",
        rel_start=int(-0.5 / Params.BIN_SIZE),
        rel_end=int(1.5 / Params.BIN_SIZE),
    )

df_trials = pyal.select_trials(df_, df.trial_name == 'trial')
df_trials = pyal.restrict_to_interval(df_trials, epoch_fun=perturb_epoch)
ccs = get_ccs_between_two_areas(df_trials, area1=areas[0], area2=areas[1], n_components=10)

fig, ax = plt.subplots(figsize=(8, 5))
x_ = np.arange(1,n_components+1)
ax.plot(x_, ccs, color='k', marker = 'o')
ax.set_ylabel('Canonical correlation')
ax.set_xlabel('Neural mode')
ax.set_title(f"CCA between {areas[0]} and {areas[1]}")

print(ccs)


### CCA with different time lags

In [ ]:
from tools.dimensionality.cca import get_ccs_between_two_areas

# Function to roll the arrays
def roll_array(arr, shift):
    return np.roll(arr, shift=shift, axis=0)  # Rolling along the time axis

df_trials = pyal.select_trials(df_, df.trial_name == 'trial')

perturb_epoch = pyal.generate_epoch_fun(
        start_point_name="idx_sol_on",
        rel_start=int(0 / Params.BIN_SIZE),
        rel_end=int(1.5 / Params.BIN_SIZE),
    )


# Shift variables, all in seconds
shift_start = -0.5
shift_end = 0.5
shift_step = 0.05
shifts = np.arange(
    start= int(shift_start / Params.BIN_SIZE), 
    stop=int(shift_end / Params.BIN_SIZE), 
    step=int(shift_step / Params.BIN_SIZE)
    )  # Example shift value
df_tmp = df_trials.copy()
area_to_be_shifted = "M1"
ccs_time_shifted = []
for shift in shifts:
    df_tmp = df_trials.copy()
    df_tmp[f"{area_to_be_shifted}_rates"] = df_tmp[f"{area_to_be_shifted}_rates"].apply(lambda arr: roll_array(arr, shift))
    df_tmp = pyal.restrict_to_interval(df_tmp, epoch_fun=perturb_epoch)
    ccs_time_shifted.append(get_ccs_between_two_areas(df_tmp, area1=areas[0], area2=areas[1], n_components=10)[0])



In [ ]:
from tools.dimensionality.cca import compute_shifted_cca_between_areas

df_trials = pyal.select_trials(df_, df.trial_name == 'trial')

ccs_time_shifted, shifts = compute_shifted_cca_between_areas(
    df_trials, 
    area_to_shift="M1", 
    area_to_compare="Dls",
    shift_step=Params.BIN_SIZE
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(shifts, ccs_time_shifted, color='k', marker = 'o')
ax.axvline(x=shifts[np.argmax(ccs_time_shifted)], color='r', linestyle='--')
ax.set_ylabel('CCA')
ax.set_xlabel('Time shift (s)')
ax.set_title(f"CCA with M1 shifted. Peak at {shifts[np.argmax(ccs_time_shifted)]:.4f}")


In [ ]:
from tools.dimensionality.cca import compute_shifted_cca_between_areas

df_trials = pyal.select_trials(df_, df.trial_name == 'trial')

ccs_time_shifted, shifts = compute_shifted_cca_between_areas(
    df_trials, 
    area_to_shift="M1", 
    area_to_compare="Dls",
    shift_step=Params.BIN_SIZE
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(shifts, ccs_time_shifted, color='k', marker = 'o')
ax.axvline(x=shifts[np.argmax(ccs_time_shifted)], color='r', linestyle='--')
ax.set_ylabel('CCA')
ax.set_xlabel('Time shift (s)')
ax.set_title(f"CCA with M1 shifted. Peak at {shifts[np.argmax(ccs_time_shifted)]:.4f}")


In [ ]:
pyal.get_time_varying_fields(df_)
pyal.get